# CH101 SPAR3D diagnostic-only run

This notebook reproduces the already-authorized SPAR3D path once to capture a short sanitized runtime failure signal. It never registers a candidate, never creates a review `.blend`, and never opens Unity or Production gates. Run it only on a Kaggle GPU runtime after the SPAR3D Hugging Face access and license Secrets are connected.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import urllib.request
from PIL import Image

CHARACTER_CODE = 'CH101'
TOOLS_REPO = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_REF = 'feature/ch101-free-ai3d-autobuild'
ART_REPO = 'https://github.com/siri2677/re-camp.git'
ART_MEDIA_BASE = 'https://media.githubusercontent.com/media/siri2677/re-camp'
ART_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
SPAR3D_REPO = 'https://github.com/Stability-AI/stable-point-aware-3d.git'
SPAR3D_COMMIT = 'fdc311b16809e6a8adc2f5a3407ebb3db1a95bd1'
SPAR3D_TEXTURE_RESOLUTION = os.environ.get('RE_CAMP_SPAR3D_TEXTURE_RESOLUTION', '512')
SPAR3D_DECODER_CHUNK_SIZE = os.environ.get('SPAR3D_DECODER_CHUNK_SIZE', '8192')
SPAR3D_ATTENTION_QUERY_CHUNK_SIZE = os.environ.get('SPAR3D_ATTENTION_QUERY_CHUNK_SIZE', '256')
CONTENT_ROOT = Path('/kaggle/working')
TOOLS_DIR = CONTENT_ROOT / 're-camp-blender'
ART_DIR = CONTENT_ROOT / 're-camp'
OUTPUT_DIR = CONTENT_ROOT / 're-camp-ai3d' / CHARACTER_CODE / 'spar3d-diagnostic'
REFERENCE_DIR = OUTPUT_DIR / 'reference-views'
PROVIDER_DIR = CONTENT_ROOT / 'provider-SPAR3D'
REPORT = OUTPUT_DIR / 'spar3d-diagnostic-report.json'
GATES = {'sourceStatus': 'AI_GENERATED_CANDIDATE_NOT_PRODUCTION', 'gateB': 'PENDING_HUMAN_REVIEW', 'unityInputAllowed': False, 'productionPromotionAllowed': False}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({'runtime': 'kaggle', 'character': CHARACTER_CODE, 'diagnosticOnly': True, **GATES})

In [ ]:
def run(command, *, cwd=None, check=True, env=None):
    command = [str(part) for part in command]
    print('RUN:', ' '.join(command))
    result = subprocess.run(command, cwd=cwd, env=env, check=False)
    if check and result.returncode:
        raise RuntimeError(f'command failed ({result.returncode})')
    return result

def load_secret(name):
    if os.environ.get(name):
        return True
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        return False
    if value:
        os.environ[name] = value
        return True
    return False

secret_names = ('HF_TOKEN', 'RE_CAMP_SPAR3D_ACCESS_ACK', 'RE_CAMP_SPAR3D_LICENSE_ACK')
loaded = [name for name in secret_names if load_secret(name)]
print({'kaggleSecretBridge': 'ENABLED', 'loadedSecretCount': len(loaded), 'secretNames': loaded, 'secretValuesRecorded': False})

if not (TOOLS_DIR / '.git').is_dir():
    run(['git', 'clone', '--depth', '1', '--branch', TOOLS_REF, TOOLS_REPO, TOOLS_DIR])
else:
    run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_REF])
    run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', f'origin/{TOOLS_REF}'])
tools_commit = subprocess.check_output(['git', '-C', str(TOOLS_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print({'toolsCommit': tools_commit, 'toolsRef': TOOLS_REF})

if not (PROVIDER_DIR / '.git').is_dir():
    run(['git', 'clone', '--no-checkout', SPAR3D_REPO, PROVIDER_DIR])
run(['git', '-C', PROVIDER_DIR, 'fetch', '--depth', '1', 'origin', SPAR3D_COMMIT])
run(['git', '-C', PROVIDER_DIR, 'checkout', '--detach', SPAR3D_COMMIT])
provider_commit = subprocess.check_output(['git', '-C', str(PROVIDER_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print({'providerCommit': provider_commit, 'providerCommitExpected': SPAR3D_COMMIT})

contract = json.loads((TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json').read_text(encoding='utf-8'))
character = next(item for item in contract['characters'] if item['character'] == CHARACTER_CODE)
relative_paths = [character['authoritativeSource'], character['generationSource']['path']] + [item['path'] for item in character.get('auxiliaryReferences', [])]
for relative_path in dict.fromkeys(relative_paths):
    destination = ART_DIR / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    # GitHub raw serves an LFS pointer for these locked PNGs; media.githubusercontent.com serves the binary LFS object.
    url = f"{ART_MEDIA_BASE}/{ART_COMMIT}/{relative_path}"
    urllib.request.urlretrieve(url, destination)
    with Image.open(destination) as downloaded_image:
        downloaded_image.verify()
ART_DIR_COMMIT = ART_COMMIT
print({'artCommit': ART_DIR_COMMIT, 'artFilesDownloaded': len(dict.fromkeys(relative_paths))})

In [ ]:
preflight = OUTPUT_DIR / 'spar3d-preflight.json'
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'colab_runtime_preflight.py', '--provider', 'spar3d', '--output', preflight])
preflight_payload = json.loads(preflight.read_text(encoding='utf-8'))
print({'status': preflight_payload['status'], 'providerPreflight': preflight_payload['providerPreflight'], **GATES})
if preflight_payload['status'] != 'READY_GPU_VISIBLE' or preflight_payload['providerPreflight'].get('heavyweightInstallAllowed') is not True:
    raise RuntimeError('SPAR3D diagnostic stopped before installation: provider preflight is not ready')

run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_reference_views.py', '--art-root', ART_DIR, '--output-dir', REFERENCE_DIR, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
reference_manifest = REFERENCE_DIR / 'reference-views-manifest.json'
setup_steps = [
    [sys.executable, '-m', 'pip', 'install', '-q', '-U', 'setuptools==69.5.1', 'wheel'],
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-r', PROVIDER_DIR / 'requirements.txt', 'flet==0.23.1'],
]
for setup_step in setup_steps:
    run(setup_step, cwd=PROVIDER_DIR)

provider_env = os.environ.copy()
provider_env['SPAR3D_DECODER_CHUNK_SIZE'] = SPAR3D_DECODER_CHUNK_SIZE
provider_env['SPAR3D_ATTENTION_QUERY_CHUNK_SIZE'] = SPAR3D_ATTENTION_QUERY_CHUNK_SIZE
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_spar3d_candidate.py', '--provider-repo', PROVIDER_DIR, '--input-image', REFERENCE_DIR / 'CH101_front.png', '--output-dir', OUTPUT_DIR / 'provider-output', '--preflight', preflight, '--output-report', REPORT, '--texture-resolution', SPAR3D_TEXTURE_RESOLUTION, '--target-count', '20000', '--diagnostic-only', '--execute'], check=False, env=provider_env)
payload = json.loads(REPORT.read_text(encoding='utf-8'))
payload['toolsCommit'] = tools_commit
payload['artCommit'] = ART_COMMIT
payload['providerCommitExpected'] = SPAR3D_COMMIT
payload['providerCommitActual'] = provider_commit
payload['referenceManifest'] = str(reference_manifest)
payload.update(GATES)
REPORT.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print({'diagnosticReport': str(REPORT), 'status': payload.get('status'), 'executionFailureDetail': payload.get('executionFailureDetail', 'NONE'), 'diagnosticMeshCount': payload.get('diagnosticMeshCount', 0), 'meshOutputs': payload.get('meshOutputs', []), 'meshSha256': payload.get('meshSha256', 'NOT_RECORDED'), 'candidateRegistered': False, **GATES})